## 사회 경제적 요인 분석 지도 (토빗 회귀)

## 토빗 분석용 행정동 단위 데이터 만들기

토빗 분석 - 영향요인 분석용이므로 행정동 단위(논문도 행정동 단위로)

| 구분      | 변수                   | 의미         |
| ------- | -------------------- | ---------- |
| 종속변수    | E2SFCA 접근성 점수 | 행정동 평균 접근성 |
| 수요 변수   | 초등학생 인구, 아동 인구 비율    | 돌봄 수요      |
| 사회경제 변수 | 지가, 학원 수(사교육), 기초수급자 수      | 지역 여건      |
| 공급 변수   | 들락날락 수, 돌봄시설 수       | 현재 공급 수준   |


In [ ]:
# 행정동 경계 불러오기
dong = gpd.read_file("읍면동/BND_ADM_DONG_PG.shp").to_crs(gdf.crs)

# 격자 중심점 만들기
grid_point = gdf.copy()
grid_point["geometry"] = grid_point.geometry.centroid

# 격자에 행정동 붙이기
grid_dong = gpd.sjoin(
    grid_point,
    dong[["ADM_CD", "ADM_NM", "geometry"]],
    how="left",
    predicate="within"
)

# 행정동별 접근성, 인구 집계
dong_access = (
    grid_dong
    .groupby(["ADM_CD", "ADM_NM"], as_index=False)
    .agg(
        access_mean=("access_e2sfca_1000", "mean"),
        child_pop=(pop_col, "sum"),
        grid_count=(pop_col, "count"),
        shortage_grid=("shortage_area", "sum")
    )
)

dong_access.head()

## 토빗 분석용 변수 전처리 및 최종 데이터 구축

앞에서 계산한 `dong_access`를 기준으로 지가, 학원 수, 기초생활보장 수급자 수를 행정동 단위로 정리한 뒤 최종 분석용 데이터 생성

## 1. 분석 기준 데이터 확인

In [ ]:
# 행정동별 접근성 결과 확인
print(dong_access.columns)
dong_access.head()

## 2. 지가 데이터 정리

In [ ]:
# 행정동 및 개별공시지가 shp 불러오기
dong_gdf = gpd.read_file("읍면동/BND_ADM_DONG_PG.shp", encoding="cp949")
land_gdf = gpd.read_file("개별공시지가/AL_D150_26_20260526.shp", encoding="cp949")

# 좌표계 통일
land_gdf = land_gdf.to_crs(dong_gdf.crs)

# 지가 컬럼 숫자형 변환
land_gdf["A9"] = pd.to_numeric(land_gdf["A9"], errors="coerce")

# 지가 필지와 행정동 공간결합
land_joined = gpd.sjoin(
    land_gdf,
    dong_gdf[["ADM_CD", "ADM_NM", "geometry"]],
    how="left",
    predicate="intersects"
)

# 행정동별 지가 요약
land_price = (
    land_joined
    .groupby(["ADM_CD", "ADM_NM"], as_index=False)
    .agg(
        mean_land_price=("A9", "mean"),
        median_land_price=("A9", "median"),
        land_count=("A9", "count")
    )
)

import os
land_price.to_csv("행정동별_지가.csv", index=False, encoding="utf-8-sig")
print("저장 완료:", os.path.exists("행정동별_지가.csv"))
land_price.head()

## 3. 학원 데이터 정리

In [ ]:
academy_raw = pd.read_csv(
    "부산광역시교육청 학원 및 교습소 현황_20260615.csv",
    encoding="utf-8-sig",
    low_memory=False
)

# 주소에서 시군구 추출
academy_raw["시군구"] = academy_raw["위치"].str.extract(
    r"부산광역시\s+(\S+(?:구|군))"
)

# 괄호 안의 법정동/읍/면/리 추출
academy_raw["읍면동"] = academy_raw["위치"].str.extract(
    r"\(([가-힣]+(?:읍|면|동\d?가?|동|리))\)"
)

# 같은 학원이 여러 교습과목으로 반복될 수 있으므로 중복 제거
academy_unique = academy_raw.drop_duplicates(
    subset=["학원명", "위치"]
).copy()

academy_unique = academy_unique.dropna(subset=["시군구", "읍면동"]).copy()

academy = (
    academy_unique
    .groupby(["시군구", "읍면동"], as_index=False)
    .agg(academy_count=("학원명", "count"))
)

academy.to_csv("행정동별_학원수.csv", index=False, encoding="utf-8-sig")
academy.head()

## 4. 학원 수를 행정동 접근성 자료에 결합

In [ ]:
import pandas as pd
import re

academy_dong_set = set(academy["읍면동"].dropna().unique())

def make_academy_match_name(adm_nm):
    # 행정동명을 학원 주소에서 추출한 법정동명과 최대한 맞추는 함수

    if adm_nm in academy_dong_set:
        return adm_nm

    special_map = {
        "동대신1동": "동대신동1가",
        "동대신2동": "동대신동2가",
        "동대신3동": "동대신동3가",
        "서대신1동": "서대신동1가",
        "서대신2동": "서대신동2가",
        "서대신3동": "서대신동3가",
        "서대신4동": "서대신동4가",
        "중앙동": "중앙동4가",
        "대청동": "대청동4가",
        "보수동": "보수동1가",
        "부평동": "부평동1가",
        "남포동": "남포동6가",
        "영선1동": "영선동1가",
        "영선2동": "영선동2가",
        "봉래1동": "봉래동3가",
        "봉래2동": "봉래동4가",
    }

    if adm_nm in special_map:
        return special_map[adm_nm]

    # 예: 화명1동 → 화명동, 광안4동 → 광안동
    candidate = re.sub(r"(.+?)\d+동$", r"\1동", adm_nm)

    if candidate in academy_dong_set:
        return candidate

    return adm_nm


dong_access["academy_match_dong"] = dong_access["ADM_NM"].apply(make_academy_match_name)

model_df = dong_access.merge(
    academy[["읍면동", "academy_count"]],
    left_on="academy_match_dong",
    right_on="읍면동",
    how="left"
)

model_df["academy_count"] = model_df["academy_count"].fillna(0)

model_df[["ADM_CD", "ADM_NM", "academy_match_dong", "academy_count"]].head()

## 5. 부산 행정동만 남기기

In [ ]:
model_df["ADM_CD"] = model_df["ADM_CD"].astype(str)

model_df_busan = model_df[
    model_df["ADM_CD"].str.startswith("21")
].copy()

print("부산 행정동 수:", len(model_df_busan))
print("학원 수 0인 행정동 수:", (model_df_busan["academy_count"] == 0).sum())


## 6. 기초생활보장 수급자 데이터 정리

In [ ]:
welfare_raw = pd.read_csv(
    "부산광역시_기초생활보장 수급자 읍면동 현황_20231231.csv",
    encoding="cp949"
)

# 2023년 자료만 사용
welfare_2023 = welfare_raw[welfare_raw["연도"] == 2023].copy()

# 구 전체 합계 행 제거
welfare_2023 = welfare_2023[
    welfare_2023["시군구"] != welfare_2023["읍면동"]
].copy()

# 명칭 변경 반영
welfare_2023["읍면동"] = welfare_2023["읍면동"].replace({
    "일광면": "일광읍"
})

welfare_cols = [
    "일반수급자 수급권자수",
    "조건부수급자 수급권자수",
    "특례수급자 수급권자수",
    "기타 수급권자수",
    "시설수급자 수급권자수"
]

for col in welfare_cols:
    welfare_2023[col] = pd.to_numeric(welfare_2023[col], errors="coerce").fillna(0)

welfare_2023["welfare_count"] = welfare_2023[welfare_cols].sum(axis=1)

welfare = welfare_2023[["시군구", "읍면동", "welfare_count"]].copy()
welfare.to_csv("행정동별_기초수급자수.csv", index=False, encoding="utf-8-sig")

welfare.head()

## 7. 기초수급자 데이터에 행정동 코드 붙이기

In [ ]:
dong_code_table = dong_access[["ADM_CD", "ADM_NM"]].copy()
dong_code_table = dong_code_table.rename(columns={"ADM_NM": "읍면동"})
dong_code_table["ADM_CD"] = dong_code_table["ADM_CD"].astype(str)

welfare_with_code = welfare.merge(
    dong_code_table,
    on="읍면동",
    how="left"
)

print("ADM_CD 안 붙은 기초수급자 행 수:", welfare_with_code["ADM_CD"].isna().sum())

welfare_with_code.to_csv(
    "행정동별_기초수급자수_ADM_CD포함.csv",
    index=False,
    encoding="utf-8-sig"
)

welfare_with_code.head()


## 8. 최종 데이터 결합

In [ ]:
import os
import pandas as pd

# 지가 데이터 불러오기
land_price = pd.read_csv("행정동별_지가.csv", encoding="utf-8-sig")

# ADM_CD 자료형 통일
model_df_busan["ADM_CD"] = model_df_busan["ADM_CD"].astype(str)
land_price["ADM_CD"] = land_price["ADM_CD"].astype(str)
welfare_with_code["ADM_CD"] = welfare_with_code["ADM_CD"].astype(str)

# 지가 데이터 결합
model_df_final = model_df_busan.merge(
    land_price[["ADM_CD", "mean_land_price", "median_land_price", "land_count"]],
    on="ADM_CD",
    how="left"
)

# 기초수급자 데이터 결합
model_df_final = model_df_final.merge(
    welfare_with_code[["ADM_CD", "welfare_count"]],
    on="ADM_CD",
    how="left"
)

# 결측값 확인
check_cols = [
    "access_mean",
    "child_pop",
    "shortage_grid",
    "academy_count",
    "welfare_count",
    "mean_land_price",
    "median_land_price",
    "land_count"
]

print("정리 전 결측값:")
print(model_df_final[check_cols].isna().sum())

# 아동 인구가 있고, 주요 변수에 결측이 없는 행정동만 남기기
model_df_final_clean = model_df_final[
    (model_df_final["child_pop"] > 0) &
    (model_df_final["access_mean"].notna()) &
    (model_df_final["welfare_count"].notna()) &
    (model_df_final["mean_land_price"].notna()) &
    (model_df_final["academy_count"].notna())
].copy()

print("\n정리 후 결측값:")
print(model_df_final_clean[check_cols].isna().sum())

# 토빗 분석용 최종 데이터 저장
model_df_final_clean.to_csv(
    "토빗분석용_행정동별_최종데이터_clean.csv",
    index=False,
    encoding="utf-8-sig"
)

print("\n최종 행 수:", len(model_df_final_clean))
print("저장 완료:", os.path.exists("토빗분석용_행정동별_최종데이터_clean.csv"))

model_df_final_clean.head()

## 토빗 분석 모형 정의

종속변수인 돌봄공백 격자 수(`shortage_grid`)는 0 이상의 값만 가지므로, 0에서 좌측 검열된 토빗 모형을 사용한다.

shortage_grid = f(아동인구, 학원 수, 기초수급자 수, 평균지가)

In [ ]:
!pip install statsmodels

In [ ]:
import numpy as np
import pandas as pd
import statsmodels.api as sm
from statsmodels.base.model import GenericLikelihoodModel
from scipy.stats import norm

class TobitModel(GenericLikelihoodModel):
    def __init__(self, endog, exog, left=0, right=np.inf, **kwds):
        self.left = left
        self.right = right
        super().__init__(endog, exog, **kwds)

    def nloglikeobs(self, params):
        beta = params[:-1]
        sigma = np.exp(params[-1])

        y = self.endog
        Xb = np.dot(self.exog, beta)

        left = self.left

        # y > left: 일반 관측값
        uncensored = y > left

        ll = np.zeros(len(y))

        # 비검열 관측치 likelihood
        ll[uncensored] = norm.logpdf(
            y[uncensored],
            loc=Xb[uncensored],
            scale=sigma
        )

        # left-censored 관측치 likelihood
        ll[~uncensored] = norm.logcdf(
            (left - Xb[~uncensored]) / sigma
        )

        return -ll

    def fit(self, start_params=None, maxiter=10000, maxfun=5000, **kwds):
        if start_params is None:
            ols_res = sm.OLS(self.endog, self.exog).fit()
            sigma_start = np.log(np.std(ols_res.resid))
            start_params = np.append(ols_res.params, sigma_start)

        return super().fit(
            start_params=start_params,
            maxiter=maxiter,
            maxfun=maxfun,
            **kwds
        )

In [ ]:
# 토빗 분석용 데이터 생성
tobit_df = model_df_final_clean.copy()

# 분석에 사용할 변수만 선택
tobit_cols = [
    "shortage_grid",
    "child_pop",
    "mean_land_price",
    "academy_count",
    "welfare_count"
]

tobit_df = tobit_df[tobit_cols].copy()

# 숫자형 변환
for col in tobit_cols:
    tobit_df[col] = pd.to_numeric(tobit_df[col], errors="coerce")

# 결측 제거
tobit_df = tobit_df.dropna().copy()

# shortage_grid는 0 이상만 사용
tobit_df = tobit_df[tobit_df["shortage_grid"] >= 0].copy()

# 로그 변환
tobit_df["ln_child_pop"] = np.log1p(tobit_df["child_pop"])
tobit_df["ln_land_price"] = np.log1p(tobit_df["mean_land_price"])
tobit_df["ln_academy"] = np.log1p(tobit_df["academy_count"])
tobit_df["ln_welfare"] = np.log1p(tobit_df["welfare_count"])

# 종속변수
y = tobit_df["shortage_grid"].values

# 독립변수
X = tobit_df[[
    "ln_child_pop",
    "ln_land_price",
    "ln_academy",
    "ln_welfare"
]]

# 상수항 추가
X = sm.add_constant(X)

# 토빗 모형 추정
tobit_model = TobitModel(y, X, left=0)
tobit_result = tobit_model.fit()

print(tobit_result.summary())

## 토빗 분석 결과 해석 기준

본 분석에서는 고수요·저접근 격자 수(`shortage_grid`)를 종속변수로 설정하였다.  
따라서 계수가 양수이면 해당 변수가 증가할수록 돌봄공백 격자 수가 증가하는 경향을 의미하고, 계수가 음수이면 돌봄공백 격자 수가 감소하는 경향을 의미한다.

(p_value < 0.05이면 통계적으로 유의)

| 변수 | 계수 방향 | 해석 |
|---|---:|---|
| `ln_child_pop` | 양수 | 아동 인구가 많은 지역일수록 돌봄 수요가 커져 돌봄공백 격자 수가 증가하는 경향 |
| `ln_child_pop` | 음수 | 아동 인구가 많은 지역에 시설 공급도 함께 배치되어 돌봄공백이 줄어드는 경향 |
| `ln_land_price` | 양수 | 지가가 높은 지역일수록 시설 입지가 제한되어 돌봄공백이 커지는 경향 |
| `ln_land_price` | 음수 | 지가가 높은 중심지일수록 기존 생활 인프라가 많아 돌봄공백이 줄어드는 경향 |
| `ln_academy` | 양수 | 학원 수가 많은 지역에도 공공 돌봄시설 공급이 충분하지 않아 돌봄공백이 커지는 경향 |
| `ln_academy` | 음수 | 사교육 인프라가 많은 지역일수록 아동 관련 시설 접근성도 좋아 돌봄공백이 줄어드는 경향 |
| `ln_welfare` | 양수 | 기초수급자 수가 많은 취약지역일수록 돌봄공백이 커지는 경향 |
| `ln_welfare` | 음수 | 취약계층이 많은 지역에 공공 돌봄서비스가 우선 배치되어 돌봄공백이 줄어드는 경향 |

## 사회경제적 요인별 공간 분포 지도

토빗 분석에 사용한 사회경제적 요인인 학원 수, 기초생활보장 수급자 수, 평균 지가를 행정동 단위 단계구분도로 시각화 → 돌봄공백 지역이 어떤 사회경제적 특성을 가진 지역과 중첩되는지 확인

In [ ]:
# ADM_CD 자료형 통일
dong_gdf["ADM_CD"] = dong_gdf["ADM_CD"].astype(str)
model_df_final_clean["ADM_CD"] = model_df_final_clean["ADM_CD"].astype(str)

# 지도용 GeoDataFrame 생성
map_gdf = dong_gdf.merge(
    model_df_final_clean,
    on=["ADM_CD", "ADM_NM"],
    how="left"
)

map_gdf.head()

## 1. 행정동별 학원 수 지도

In [ ]:
import matplotlib.pyplot as plt

# 한글 폰트 설정
plt.rcParams["font.family"] = "Malgun Gothic"
plt.rcParams["axes.unicode_minus"] = False

# 부산 행정동만 선택
dong_gdf["ADM_CD"] = dong_gdf["ADM_CD"].astype(str)
model_df_final_clean["ADM_CD"] = model_df_final_clean["ADM_CD"].astype(str)

busan_dong_gdf = dong_gdf[dong_gdf["ADM_CD"].str.startswith("21")].copy()
busan_df = model_df_final_clean[
    model_df_final_clean["ADM_CD"].str.startswith("21")
].copy()

# 지도용 데이터 결합
busan_map_gdf = busan_dong_gdf.merge(
    busan_df,
    on=["ADM_CD", "ADM_NM"],
    how="left"
)

# 지도 그리기
fig, ax = plt.subplots(figsize=(9, 10))

# 부산 지형 배경
busan_map_gdf.plot(
    ax=ax,
    color="#eeeeee",
    edgecolor="#f5f5f5",
    linewidth=0.2
)

# 학원 수 단계구분도
busan_map_gdf.plot(
    column="academy_count",
    cmap="YlGnBu",
    scheme="quantiles",
    k=5,
    legend=True,
    edgecolor="#f2f2f2",
    linewidth=0.15,
    ax=ax,
    missing_kwds={
        "color": "#eeeeee",
        "label": "자료 없음"
    }
)

# 부산 지형만 딱 보이도록 범위 설정
minx, miny, maxx, maxy = busan_map_gdf.total_bounds

x_margin = (maxx - minx) * 0.02
y_margin = (maxy - miny) * 0.02

ax.set_xlim(minx - x_margin, maxx + x_margin)
ax.set_ylim(miny - y_margin, maxy + y_margin)

ax.set_title("부산광역시 행정동별 학원 수", fontsize=16, pad=15)
ax.set_axis_off()
ax.set_aspect("equal")

plt.savefig(
    "부산광역시_행정동별_학원수_지도.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()

## 2. 행정동별 기초수급자 수 지도

In [ ]:
fig, ax = plt.subplots(figsize=(9, 10))

busan_map_gdf.plot(
    ax=ax,
    color="#eeeeee",
    edgecolor="#f5f5f5",
    linewidth=0.2
)

busan_map_gdf.plot(
    column="welfare_count",
    cmap="YlGnBu",
    scheme="quantiles",
    k=5,
    legend=True,
    edgecolor="#f2f2f2",
    linewidth=0.15,
    ax=ax,
    missing_kwds={
        "color": "#eeeeee",
        "label": "자료 없음"
    }
)

minx, miny, maxx, maxy = busan_map_gdf.total_bounds
x_margin = (maxx - minx) * 0.02
y_margin = (maxy - miny) * 0.02

ax.set_xlim(minx - x_margin, maxx + x_margin)
ax.set_ylim(miny - y_margin, maxy + y_margin)

ax.set_title("부산광역시 행정동별 기초생활보장 수급자 수", fontsize=16, pad=15)
ax.set_axis_off()
ax.set_aspect("equal")

plt.savefig(
    "부산광역시_행정동별_기초수급자수_지도.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()

## 3. 행정동별 평균 지가 지도

In [ ]:
busan_map_gdf["mean_land_price_million"] = (
    busan_map_gdf["mean_land_price"] / 1_000_000
)

fig, ax = plt.subplots(figsize=(9, 10))

busan_map_gdf.plot(
    ax=ax,
    color="#eeeeee",
    edgecolor="#f5f5f5",
    linewidth=0.2
)

busan_map_gdf.plot(
    column="mean_land_price_million",
    cmap="YlGnBu",
    scheme="quantiles",
    k=5,
    legend=True,
    edgecolor="#f2f2f2",
    linewidth=0.15,
    ax=ax,
    missing_kwds={
        "color": "#eeeeee",
        "label": "자료 없음"
    }
)

minx, miny, maxx, maxy = busan_map_gdf.total_bounds
x_margin = (maxx - minx) * 0.02
y_margin = (maxy - miny) * 0.02

ax.set_xlim(minx - x_margin, maxx + x_margin)
ax.set_ylim(miny - y_margin, maxy + y_margin)

ax.set_title("부산광역시 행정동별 평균 지가", fontsize=16, pad=15)
ax.set_axis_off()
ax.set_aspect("equal")

plt.savefig(
    "부산광역시_행정동별_평균지가_지도.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()

## 4. 사회경제적 취약성 보조지도

## 1. 사회경제적 취약성 점수 만들기

In [ ]:
from sklearn.preprocessing import MinMaxScaler

# 사회경제적 취약성 점수에 사용할 변수
vul_cols = [
    "welfare_count",    # 기초수급자 수: 많을수록 취약
    "shortage_grid",    # 돌봄공백 격자 수: 많을수록 취약
    "academy_count"     # 학원 수: 적을수록 취약
]

vul_df = model_df_final_clean[vul_cols].copy()

# 숫자형 변환
for col in vul_cols:
    vul_df[col] = pd.to_numeric(vul_df[col], errors="coerce")

# 결측 제거 또는 0 처리
vul_df = vul_df.fillna(0)

# 0~1 사이로 표준화
scaler = MinMaxScaler()
scaled = scaler.fit_transform(vul_df)

model_df_final_clean["welfare_s"] = scaled[:, 0]
model_df_final_clean["shortage_s"] = scaled[:, 1]
model_df_final_clean["academy_s"] = scaled[:, 2]

# 사회경제적 취약성 점수
# 기초수급자 많음 + 돌봄공백 많음 + 학원 수 적음
model_df_final_clean["socio_vulnerability"] = (
    model_df_final_clean["welfare_s"] +
    model_df_final_clean["shortage_s"] +
    (1 - model_df_final_clean["academy_s"])
)

model_df_final_clean[[
    "ADM_CD",
    "ADM_NM",
    "welfare_count",
    "shortage_grid",
    "academy_count",
    "socio_vulnerability"
]].sort_values("socio_vulnerability", ascending=False).head(10)

# 지도용 데이터 만들기
# ADM_CD 문자형 통일
dong_gdf["ADM_CD"] = dong_gdf["ADM_CD"].astype(str)
model_df_final_clean["ADM_CD"] = model_df_final_clean["ADM_CD"].astype(str)

# 지도용 데이터 생성
map_gdf = dong_gdf.merge(
    model_df_final_clean,
    on=["ADM_CD", "ADM_NM"],
    how="left"
)

map_gdf.head()

## 2. 사회경제적 취약성 보조지도 그리기

In [ ]:
import matplotlib.pyplot as plt

# 한글 폰트 설정
plt.rcParams["font.family"] = "Malgun Gothic"
plt.rcParams["axes.unicode_minus"] = False

# ADM_CD 문자형 통일
map_gdf["ADM_CD"] = map_gdf["ADM_CD"].astype(str)

# 부산광역시 행정동만 선택
busan_map_gdf = map_gdf[map_gdf["ADM_CD"].str.startswith("21")].copy()

fig, ax = plt.subplots(figsize=(9, 10))

# 부산 지형 배경
busan_map_gdf.plot(
    ax=ax,
    color="#eeeeee",
    edgecolor="#f5f5f5",
    linewidth=0.2
)

# 사회경제적 취약성 지도
busan_map_gdf.plot(
    column="socio_vulnerability",
    cmap="YlGnBu",
    scheme="quantiles",
    k=5,
    legend=True,
    edgecolor="#f2f2f2",
    linewidth=0.15,
    ax=ax,
    missing_kwds={
        "color": "#eeeeee",
        "label": "자료 없음"
    }
)

# 부산 지형만 딱 보이도록 범위 설정
minx, miny, maxx, maxy = busan_map_gdf.total_bounds
x_margin = (maxx - minx) * 0.02
y_margin = (maxy - miny) * 0.02

ax.set_xlim(minx - x_margin, maxx + x_margin)
ax.set_ylim(miny - y_margin, maxy + y_margin)

ax.set_title("부산광역시 사회경제적 돌봄 취약성 보조지도", fontsize=16, pad=15)
ax.set_axis_off()
ax.set_aspect("equal")

plt.savefig(
    "부산광역시_사회경제적_돌봄취약성_보조지도.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()

## 3. 상위 25% 취약지역 지도

In [ ]:
import matplotlib.pyplot as plt

# 한글 폰트 설정
plt.rcParams["font.family"] = "Malgun Gothic"
plt.rcParams["axes.unicode_minus"] = False

# 부산 행정동만 사용
busan_map_gdf["ADM_CD"] = busan_map_gdf["ADM_CD"].astype(str)

# 사회경제적 취약성 상위 25% 기준값 계산
threshold_75 = busan_map_gdf["socio_vulnerability"].quantile(0.75)

# 상위 25% 여부 표시
busan_map_gdf["is_top25_vul"] = (
    busan_map_gdf["socio_vulnerability"] >= threshold_75
)

print("상위 25% 기준값:", threshold_75)
print("상위 25% 행정동 수:", busan_map_gdf["is_top25_vul"].sum())

# 상위 25% 데이터만 따로 분리
top25_gdf = busan_map_gdf[busan_map_gdf["is_top25_vul"]].copy()

# 지도 그리기
fig, ax = plt.subplots(figsize=(9, 10))

# 1. 전체 부산 행정동은 연회색으로 표시
busan_map_gdf.plot(
    ax=ax,
    color="#eeeeee",
    edgecolor="#f7f7f7",
    linewidth=0.2
)

# 2. 상위 25% 취약지역만 색으로 표시
top25_gdf.plot(
    column="socio_vulnerability",
    cmap="YlGnBu",
    scheme="quantiles",
    k=5,
    legend=True,
    edgecolor="white",
    linewidth=0.25,
    ax=ax
)

# 부산 범위에 맞게 축 조정
minx, miny, maxx, maxy = busan_map_gdf.total_bounds
x_margin = (maxx - minx) * 0.02
y_margin = (maxy - miny) * 0.02

ax.set_xlim(minx - x_margin, maxx + x_margin)
ax.set_ylim(miny - y_margin, maxy + y_margin)

ax.set_title("부산광역시 사회경제적 돌봄 취약성 상위 25% 지역", fontsize=15, pad=15)
ax.set_axis_off()
ax.set_aspect("equal")

plt.savefig(
    "부산광역시_사회경제적_돌봄취약성_상위25퍼센트_지도.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()

## 고수요 - 저접근 지역 안에서 위성형 후보지 찾기

## 1. 기존 데이터 확인 및 정리

In [ ]:
import pandas as pd
import geopandas as gpd

# 후보시설 불러오기
library_df = pd.read_csv("부산_도서관_좌표.csv", encoding="utf-8-sig")
admin_df = pd.read_csv("부산_행정복지센터_API.csv", encoding="utf-8-sig")

# 컬럼 확인
print("도서관 컬럼:", library_df.columns)
print("행정복지센터 컬럼:", admin_df.columns)

In [ ]:
# 1. 도서관 데이터 정리
library_df["lat"] = pd.to_numeric(library_df["lat"], errors="coerce")
library_df["lng"] = pd.to_numeric(library_df["lng"], errors="coerce")

library_df = library_df.dropna(subset=["name", "lat", "lng"]).copy()

library_gdf = gpd.GeoDataFrame(
    library_df,
    geometry=gpd.points_from_xy(library_df["lng"], library_df["lat"]),
    crs="EPSG:4326"
)

library_gdf["candidate_type"] = "도서관"
library_gdf["address"] = library_gdf["address"]

In [ ]:
# 2. 행정복지센터 데이터 정리
admin_df["lat"] = pd.to_numeric(admin_df["lat"], errors="coerce")
admin_df["lng"] = pd.to_numeric(admin_df["lng"], errors="coerce")

admin_df = admin_df.dropna(subset=["dept", "lat", "lng"]).copy()

# 컬럼명 통일
admin_df = admin_df.rename(columns={
    "dept": "name",
    "addrRoad": "address"
})

admin_gdf = gpd.GeoDataFrame(
    admin_df,
    geometry=gpd.points_from_xy(admin_df["lng"], admin_df["lat"]),
    crs="EPSG:4326"
)

admin_gdf["candidate_type"] = "행정복지센터"

In [ ]:
# 3. 기존 분석 격자 gdf 좌표계에 맞추기
library_gdf = library_gdf.to_crs(gdf.crs)
admin_gdf = admin_gdf.to_crs(gdf.crs)

In [ ]:
# 4. 도서관 + 행정복지센터 후보시설 합치기
candidate_gdf = pd.concat([
    library_gdf[["name", "candidate_type", "gugun", "address", "geometry"]],
    admin_gdf[["name", "candidate_type", "gugun", "address", "geometry"]]
], ignore_index=True)

candidate_gdf = gpd.GeoDataFrame(
    candidate_gdf,
    geometry="geometry",
    crs=gdf.crs
)

print(candidate_gdf.shape)
candidate_gdf.head()

## 2. 고수요 - 저접근성 지역 추출 및 수요 계산

In [ ]:
# 5. 고수요-저접근성 지역만 추출
shortage_gdf = gdf[gdf["gap_type"] == "고수요-저접근성"].copy()

# 격자 중심점으로 계산
shortage_point = shortage_gdf.copy()
shortage_point["geometry"] = shortage_point.geometry.centroid

In [ ]:
# 6. 후보지별 750m 이내 고수요-저접근 수요 계산
pop_col = "val"
BUFFER_DIST = 750

candidate_gdf["candidate_id"] = range(len(candidate_gdf))

coverage_list = []

for idx, cand in candidate_gdf.iterrows():
    buffer_geom = cand.geometry.buffer(BUFFER_DIST)

    covered = shortage_point[
        shortage_point.geometry.within(buffer_geom)
    ].copy()

    coverage_list.append({
        "candidate_id": cand["candidate_id"],
        "name": cand["name"],
        "candidate_type": cand["candidate_type"],
        "gugun": cand["gugun"],
        "address": cand["address"],
        "covered_child_pop": covered[pop_col].sum(),
        "covered_grid_count": len(covered)
    })

coverage_df = pd.DataFrame(coverage_list)

candidate_result = candidate_gdf.merge(
    coverage_df,
    on=["candidate_id", "name", "candidate_type", "gugun", "address"],
    how="left"
)

candidate_result = candidate_result.sort_values(
    "covered_child_pop",
    ascending=False
)

candidate_result[[
    "name",
    "candidate_type",
    "gugun",
    "address",
    "covered_child_pop",
    "covered_grid_count"
]].head(20)

## 3. 기존 들락날락과 가까운 후보 제외

In [ ]:
# 7. 기존 들락날락과 너무 가까운 후보 제외
dlnl_union = dlnl_gdf.to_crs(candidate_result.crs).geometry.union_all()

candidate_result["dist_to_nearest_dlnl"] = (
    candidate_result.geometry.distance(dlnl_union)
)

candidate_filtered = candidate_result[
    (candidate_result["covered_child_pop"] > 0) &
    (candidate_result["dist_to_nearest_dlnl"] >= 500)
].copy()

candidate_filtered = candidate_filtered.sort_values(
    "covered_child_pop",
    ascending=False
)

satellite_selected = candidate_filtered.head(10).copy()

satellite_selected[[
    "name",
    "candidate_type",
    "gugun",
    "address",
    "covered_child_pop",
    "covered_grid_count",
    "dist_to_nearest_dlnl"
]]

In [ ]:
print("위성형 후보지 총 개수:", candidate_filtered.shape[0])

## 부산 고수요-저접근 지역과 들락날락 거점-위성형 후보지 지도

In [ ]:
fig, ax = plt.subplots(figsize=(12, 12))

# --------------------------------------------------
# 1. 전체 격자 배경
# --------------------------------------------------
gdf.plot(
    ax=ax,
    color="#F2F2F2",
    edgecolor="#E6E6E6",
    linewidth=0.03,
    alpha=0.5
)

# --------------------------------------------------
# 2. 고수요-저접근성 부족지역 강조
# --------------------------------------------------
gdf[gdf["gap_type"] == "고수요-저접근성"].plot(
    ax=ax,
    color="#D73027",
    edgecolor="#A50026",
    linewidth=0.08,
    alpha=0.85,
    label="고수요-저접근 지역",
    zorder=5
)

# --------------------------------------------------
# 3. 기존 들락날락 표시
# --------------------------------------------------
dlnl_gdf.to_crs(gdf.crs).plot(
    ax=ax,
    color="green",
    markersize=10,
    marker="o",
    alpha=0.9,
    label="기존 들락날락",
    zorder=10
)

# --------------------------------------------------
# 4. 위성형 후보를 유형별로 분리
# --------------------------------------------------
satellite_selected = satellite_selected.to_crs(gdf.crs)

library_selected = satellite_selected[
    satellite_selected["candidate_type"] == "도서관"
].copy()

admin_selected = satellite_selected[
    satellite_selected["candidate_type"] == "행정복지센터"
].copy()

# --------------------------------------------------
# 5. 도서관 후보 표시
# --------------------------------------------------
if not library_selected.empty:
    library_selected.plot(
        ax=ax,
        color="blue",
        markersize=90,
        marker="*",
        edgecolor="white",
        linewidth=0.5,
        alpha=1.0,
        label="위성형 후보(도서관)",
        zorder=11
    )

# --------------------------------------------------
# 6. 행정복지센터 후보 표시
# --------------------------------------------------
if not admin_selected.empty:
    admin_selected.plot(
        ax=ax,
        color="pink",
        markersize=70,
        marker="^",
        edgecolor="white",
        linewidth=0.5,
        alpha=1.0,
        label="위성형 후보(행정복지센터)",
        zorder=12
    )

# --------------------------------------------------
# 8. 지도 범위 설정
# --------------------------------------------------
minx, miny, maxx, maxy = gdf.total_bounds
ax.set_xlim(minx, maxx)
ax.set_ylim(miny, maxy)

# --------------------------------------------------
# 9. 제목 / 범례 / 축 제거
# --------------------------------------------------
ax.set_title(
    "부산 고수요-저접근 지역과 들락날락 거점-위성형 후보지",
    fontsize=15
)

ax.legend(
    loc="center left",
    bbox_to_anchor=(1.02, 0.5),
    frameon=True
)

ax.set_axis_off()
plt.tight_layout()

plt.savefig(
    "부산_고수요_저접근_들락날락_거점_위성형_지도.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()

In [ ]:
print("지도에 표시된 위성형 후보지 수:", len(satellite_selected))